# Fine-tuning a masked language model + Работа с [HuggingFace](https://huggingface.co/) (Google Colab)
*Материал заимствован с LLM Course от HuggingFace, который является стандартом в изучении LLM и рекомендуется всем заинтересованным студентам в качестве дополнительного ресурса*

**Fine-tuning** (тонкая настройка) — это процесс дообучения предварительно обученной модели под конкретную задачу. Вместо того чтобы обучать модель с нуля, что требует огромных объёмов данных и вычислительных ресурсов, fine-tuning позволяет использовать знания, уже закодированные в весах модели, и «настроить» их под конкретные нужды.

**Masked language modeling** - задача, в которой модель предсказывает пропущенное слово в предложении ("замаскированное")




masked_modeling.svg


Для многих приложений NLP, использующих трансформерные модели, вы можете просто взять предварительно подготовленную модель из Hugging Face Hub и настроить ее непосредственно на ваших данных для решения текущей задачи. При условии, что корпус, используемый для предварительной подготовки, не слишком отличается от корпуса, используемого для точной настройки, **трансферное обучение (transfer learning)** обычно дает хорошие результаты. Для transfer learning характерно то, что мы замораживаем все слои, кроме последних (головы), в процессе дообучения.

Однако в некоторых случаях вам потребуется сначала файнтьюнить языковые модели на ваших данных, прежде чем обучать "голову" трансформера для конкретной задачи. Например, если ваш набор данных содержит юридические контракты или научные статьи, то такая простая модель, как BERT, обычно обрабатывает слова в вашем корпусе, относящиеся к предметной области, как редкие маркеры, и результаты могут быть неудовлетворительными. Путем файнтьюнинга языковой модели на основе данных, относящихся к предметной области, вы можете повысить производительность многих последующих задач, что означает, что вам обычно нужно выполнить этот шаг только один раз. В данном случае мы замораживаем только первые слои или не замораживаем их вообще и дообучаем все слои на конкретных данных.

Этот процесс файнтьюнинга предварительно подготовленной языковой модели на основе данных, относящихся к предметной области, обычно называется domain adaptation (адаптацией к предметной области).


## Выбор предварительно обученной модели для предсказания пропущенного слова

Несмотря на то, что модели семейства BERT и RoBERTa являются наиболее загружаемыми, мы будем использовать модель под названием DistilBERT, которая может быть обучена намного быстрее с минимальными потерями в производительности. Эта модель была обучена с использованием специальной методики, называемой дистилляцией знаний, при которой большая “модель учителя”, такая как BERT, используется для обучения “модели ученика”, которая имеет гораздо меньше параметров.

In [ ]:
# Загрузка модели
from transformers import AutoModelForMaskedLM

model_checkpoint = "distilbert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)


Мы можем увидеть, сколько параметров имеет модель, вызвав метод num_parameters():

In [ ]:
distilbert_num_parameters = model.num_parameters() / 1_000_000
print(f"'>>> DistilBERT number of parameters: {round(distilbert_num_parameters)}M'")
print(f"'>>> BERT number of parameters: 110M'")

Имея около 67 миллионов параметров, DistilBERT примерно в два раза меньше, чем базовая модель BERT, что примерно в два раза ускоряет обучение! Давайте теперь посмотрим, какие типы токенов по прогнозам этой модели являются наиболее вероятными дополнениями к небольшому сообщению:


In [ ]:
text = "This is a great [MASK]."

Мы можем представить множество вариантов использования лексемы [MASK], таких как “day”, “ride” или “painting”. Для предварительно обученных моделей предсказания зависят от того, на каком массиве данных была обучена модель, поскольку она учится распознавать статистические закономерности, присутствующие в данных. Как и BERT, DistilBERT был предварительно обучен работе с наборами данных английской Википедии и BookCorpus, поэтому мы ожидаем, что прогнозы для [MASK] будут отражать этот факт. Чтобы предсказать маску, нам нужен токенизатор DistilBERT для получения входных данных для модели, так что давайте также загрузим его из хаба:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

С помощью токенизатора и модели мы теперь можем передать наш текстовый пример модели, и распечатать 5 лучших кандидатов:

In [ ]:
import torch

inputs = tokenizer(text, return_tensors="pt")
token_logits = model(**inputs).logits
# Find the location of [MASK] and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
# Pick the [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(f"'>>> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

Из результатов видно, что прогнозы модели относятся к повседневным терминам, что, пожалуй, неудивительно, учитывая, что она основана на английской Википедии. Давайте посмотрим, как можно изменить эту область на что-то более узконаправленное, например, отзывы о фильмах.

## Данные

Чтобы продемонстрировать адаптацию к предметной области, мы будем использовать знаменитый большой датасет о просмотрах фильмов (сокращенно IMDb), который представляет собой набор обзоров фильмов, часто используемых для сравнения моделей анализа настроений. Доработав DistilBERT в этом корпусе, мы ожидаем, что языковая модель адаптирует свой словарный запас на основе фактических данных Википедии, на которых она была предварительно обучена, к более субъективным элементам обзоров фильмов. Мы можем получить данные из Hugging Face Hub с помощью функции load_dataset():

In [ ]:
from datasets import load_dataset

imdb_dataset = load_dataset("imdb")
imdb_dataset

Мы видим, что разделы "обучение" и "тест" содержат по 25 000 отзывов, в то время как раздел без маркировки, называемый "unsupervised", содержит 50 000 отзывов. Давайте рассмотрим несколько примеров, чтобы получить представление о том, с каким текстом мы имеем дело. Мы объединим функции Dataset.shuffle() и Dataset.select() для создания случайной выборки:

In [ ]:
sample = imdb_dataset["train"].shuffle(seed=42).select(range(3))

for row in sample:
    print(f"\n'>>> Review: {row['text']}'")
    print(f"'>>> Label: {row['label']}'")

Хотя нам не понадобятся метки для языкового моделирования, мы уже можем видеть, что 0 означает отрицательный отзыв, в то время как 1 соответствует положительному.

### **Задание**:

Создайте случайную выборку из unsupervised и убедитесь, что метки не равны ни 0, ни 1. Кроме того, вы можете проверить, что метки в обучающей и тестовой выборках действительно равны 0 или 1. Это полезная проверка, которую должен выполнять каждый специалист по обработке естественного языка в начале нового проекта!

In [ ]:
...

Теперь, когда мы бегло ознакомились с данными, давайте приступим к их подготовке для языкового моделирования. Как мы увидим, необходимо выполнить несколько дополнительных шагов по сравнению с задачами классификации текстов.

## Предварительная обработка данных

Первым этапом предварительной обработки является объединение всех примеров, а затем разбиение всего корпуса на фрагменты одинакового размера. Это сильно отличается от нашего обычного подхода, когда мы просто выделяем отдельные примеры. Зачем объединять все вместе? Причина в том, что отдельные примеры могут быть обрезаны, если они будут слишком длинными, и это приведет к потере информации, которая может быть полезна для моделирования языка.

Для начала мы токенизируем наш корпус с помощью "быстрого" токенизатора. Медленные токенизаторы - это те, которые написаны на Python в библиотеке Transformers, в то время как быстрые версии - это те, которые предоставляются токенизаторами, написанными на Rust.

Мы также используем идентификаторы слов - input_ids, поскольку они понадобятся нам позже для маскировки слов. Мы воплотим это в простую функцию, и затем удалим столбцы text и label, поскольку они нам больше не нужны:



In [ ]:
def tokenize_function(examples):
    result = tokenizer(examples["text"])
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result


# Use batched=True to activate fast multithreading!
tokenized_datasets = imdb_dataset.map(
    tokenize_function, batched=True, remove_columns=["text", "label"]
)
tokenized_datasets

Поскольку DistilBERT - это модель, подобная BERT, мы можем видеть, что закодированные тексты состоят из input_id и attention_mask, и добавленных нами word_id.

Теперь, когда мы распределили наши обзоры фильмов по категориям, следующий шаг - сгруппировать их все вместе и разделить результат на фрагменты. Но насколько большими должны быть эти фрагменты? В конечном счете, это будет зависеть от объема доступной памяти графического процессора, но хорошей отправной точкой будет посмотреть, каков максимальный размер контекста модели. Это можно определить, проверив атрибут model_max_length в токенизаторе:

In [ ]:
tokenizer.model_max_length

Это значение получено из файла tokenizer_config.json, связанного с контрольной точкой; в этом случае мы видим, что размер контекста равен 512 токенам, как и в случае с BERT.

Итак, чтобы провести наши эксперименты с графическими процессорами, подобными тем, что есть в Google Colab, мы выберем что-нибудь поменьше, способное поместиться в памяти:

In [ ]:
chunk_size = 128

Обратите внимание, что использование небольшого размера фрагмента может быть вредным в реальных сценариях, поэтому вам следует использовать размер, соответствующий задаче, к которой вы будете применять свою модель.

Чтобы показать, как работает конкатенация, давайте возьмем несколько отзывов из нашего обучающего набора токенов и выведем количество токенов в каждом отзыве:

In [ ]:
# При разбиении на части создается список списков для каждого объекта
tokenized_samples = tokenized_datasets["train"][:3]

for idx, sample in enumerate(tokenized_samples["input_ids"]):
    print(f"'>>> Review {idx} length: {len(sample)}'")

Затем мы можем объединить все эти примеры с помощью словаря:

In [ ]:
concatenated_examples = {
    k: sum(tokenized_samples[k], []) for k in tokenized_samples.keys()
}
total_length = len(concatenated_examples["input_ids"])
print(f"'>>> Concatenated reviews length: {total_length}'")

Отлично, общая длина соответствует действительности. Теперь давайте разделим объединенные обзоры на фрагменты с размером, заданным параметром chunk_size. Для этого мы перебираем объекты в concatenated_examples и используем список для создания фрагментов каждого объекта. В результате получается словарь фрагментов для каждого объекта:

In [ ]:
chunks = {
    k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
    for k, t in concatenated_examples.items()
}

for chunk in chunks["input_ids"]:
    print(f"'>>> Chunk length: {len(chunk)}'")

Как вы можете видеть в этом примере, последний фрагмент, как правило, будет меньше максимального размера фрагмента. Существуют две основные стратегии решения этой проблемы:

* Удалить последний фрагмент, если он меньше, чем заданный размер фрагмента.
* Увеличивать последний фрагмент до тех пор, пока его длина не станет равной размеру фрагмента.

Здесь мы воспользуемся первым подходом, поэтому давайте объединим всю описанную выше логику в единую функцию, которую мы можем применить к нашим маркированным наборам данных:


In [ ]:
def group_texts(examples):
    # Concatenate all texts
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    # Compute length of concatenated texts
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the last chunk if it's smaller than chunk_size
    total_length = (total_length // chunk_size) * chunk_size
    # Split by chunks of max_len
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }
    # Create a new labels column
    result["labels"] = result["input_ids"].copy()
    return result

Обратите внимание, что на последнем шаге group_texts() мы создаем новый столбец labels, который является копией столбца input_ids. Как мы вскоре увидим, это связано с тем, что при языковом моделировании цель состоит в том, чтобы предсказать случайно замаскированные токены во входном пакете, и, создавая столбец меток, мы предоставляем ground truth для нашей языковой модели.

Давайте теперь применим group_texts() к нашим размеченным наборам данных, используя функцию Dataset.map():

In [ ]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)
lm_datasets

Вы можете видеть, что группировка и последующее разбиение текстов на фрагменты позволило получить гораздо больше примеров, чем наши первоначальные 25 000 из обучающей и тестовой выборок. Это связано с тем, что теперь у нас есть примеры, включающие смежные лексемы, которые охватывают несколько примеров из исходного корпуса. Вы можете увидеть это в явном виде, поискав специальные токены [SEP] и [CLS] в одном из блоков:

In [ ]:
tokenizer.decode(lm_datasets["train"][104]["input_ids"])

В этом примерах с токенами [SEP] и [CLS]  вы можете увидеть две перекрывающиеся рецензии на фильмы.

Давайте также посмотрим, как выглядят метки классов для маскированного языкового моделирования:

In [ ]:
tokenizer.decode(lm_datasets["train"][104]["labels"])

Как и ожидалось от нашей функции group_texts(), описанной выше, это выглядит идентично расшифрованным input_ids — но тогда как наша модель может что-либо узнать? Мы упускаем ключевой шаг: нужно вставить токены [MASK] в произвольные позиции во входных данных! Давайте посмотрим, как мы можем сделать это "на лету" во время тонкой настройки с помощью специального сборщика данных.



## Файнтьюнинг DistilBERT с помощью Accelerate

Нам нужен специальный инструмент сортировки данных, который может случайным образом маскировать некоторые лексемы в каждой партии текстов. К счастью, Transformers поставляется в комплекте со специальным сборщиком данных для языкового моделирования как раз для этой задачи. Нам просто нужно передать ему токенизатор и аргумент mlm_probability, который указывает, какую часть токенов следует маскировать. Мы выберем 15%, это сумма, используемая для BERT и часто встречающаяся в литературе:

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

Чтобы увидеть, как работает случайное маскирование, давайте отправим несколько примеров в сортировщик данных. Поскольку он ожидает список dict, где каждый dict представляет собой отдельный фрагмент непрерывного текста, мы сначала выполняем итерацию по набору данных, прежде чем отправлять пакет в средство сортировки. Мы удаляем ключ "word_ids" для этого сборщика данных, поскольку он его не ожидает:

In [ ]:
samples = [lm_datasets["train"][i] for i in range(2)]
for sample in samples:
    _ = sample.pop("word_ids")

for chunk in data_collator(samples)["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")

Мы видим, что маркер [MASK] был случайным образом вставлен в разные места нашего текста. Это будут токены, которые наша модель должна будет предсказать во время обучения — и прелесть сборщика данных в том, что он будет рандомизировать вставку [МАСКИ] в каждом батче!

### **Задание**
Запустите приведенный выше фрагмент кода несколько раз, чтобы увидеть, как происходит случайная маскировка.


Одним из побочных эффектов случайной маскировки является то, что наши оценочные показатели не будут детерминированными, поскольку мы используем один и тот же сборщик данных для тренировочных и тестовых наборов. Позже, когда мы рассмотрим точную настройку с помощью Accelerate, мы увидим, как мы можем использовать гибкость собственного evaluation loop, чтобы ограничить случайность.

При обучении моделей яыкового моделирования можно маскировать целые слова вместе, а не только отдельные лексемы. Если мы хотим использовать маскировку целого слова, нам нужно будет создать сборщик данных самостоятельно. Сборщик данных - это просто функция, которая берет список текстов и преобразует их в батч, так что давайте сделаем это прямо сейчас! Мы будем использовать идентификаторы слов, вычисленные ранее, чтобы сопоставить индексы слов и соответствующие лексемы, затем случайным образом определим, какие слова следует замаскировать, и применим эту маску к входным данным. Обратите внимание, что все метки равны -100, за исключением тех, которые соответствуют маскируемым словам.

In [ ]:
import collections
import numpy as np

from transformers import default_data_collator

wwm_probability = 0.2


def whole_word_masking_data_collator(features):
    for feature in features:
        word_ids = feature.pop("word_ids")

        # Create a map between words and corresponding token indices
        mapping = collections.defaultdict(list)
        current_word_index = -1
        current_word = None
        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                if word_id != current_word:
                    current_word = word_id
                    current_word_index += 1
                mapping[current_word_index].append(idx)

        # Randomly mask words
        mask = np.random.binomial(1, wwm_probability, (len(mapping),))
        input_ids = feature["input_ids"]
        labels = feature["labels"]
        new_labels = [-100] * len(labels)
        for word_id in np.where(mask)[0]:
            word_id = word_id.item()
            for idx in mapping[word_id]:
                new_labels[idx] = labels[idx]
                input_ids[idx] = tokenizer.mask_token_id
        feature["labels"] = new_labels

    return default_data_collator(features)

Мы можем опробовать его на тех же образцах, что и раньше:

In [ ]:
samples = [lm_datasets["train"][i] for i in range(2)]
batch = whole_word_masking_data_collator(samples)

for chunk in batch["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")

Теперь, когда у нас есть два сборщика данных, остальные этапы файнтьюнинга являются стандартными. Обучение в Google Colab может занять некоторое время, поэтому сначала мы сократим размер обучающего набора до нескольких тысяч примеров. При этом мы все равно получим довольно приличную языковую модель! Быстрый способ уменьшить выборку набора данных в Datasets - это использовать функцию Dataset.train_test_split():

In [ ]:
train_size = 10_000
test_size = int(0.1 * train_size)

downsampled_dataset = lm_datasets["train"].train_test_split(
    train_size=train_size, test_size=test_size, seed=42
)
downsampled_dataset

Это автоматически создало новые разделы для обучения и тестирования, при этом размер обучающего набора был установлен равным 10 000 примеров, а для проверки — 10% от этого значения - не стесняйтесь увеличивать его, если у вас мощный графический процессор!

Следующее, что нам нужно сделать, это войти в Hugging Face Hub. Регстрируетесь на сайте Hugging Face, создаёте токен для входа и вставляете его в появившееся окно ниже. Вы можете сделать это с помощью следующей служебной функции:

In [ ]:
from huggingface_hub import notebook_login

notebook_login()



---

Мы уже увидели, что DataCollatorForLanguageModeling применяет случайную маскировку при каждой оценке, поэтому мы заметим некоторые колебания в наших метриках при каждом запуске обучения.

Один из способов устранить этот источник случайности - применить маскировку один раз ко всему набору тестов, а затем использовать средство сортировки данных по умолчанию в Transformers для сбора батчей во время оценки. Чтобы увидеть, как это работает, давайте реализуем простую функцию, которая применяет маскировку к пакету, аналогично DataCollatorForLanguageModeling:

In [ ]:
def insert_random_mask(batch):
    features = [dict(zip(batch, t)) for t in zip(*batch.values())]
    masked_inputs = data_collator(features)
    # Create a new "masked" column for each column in the dataset
    return {"masked_" + k: v.numpy() for k, v in masked_inputs.items()}

Далее мы применим эту функцию к нашему тестовому набору данных и удалим незамаскированные столбцы, чтобы заменить их замаскированными. Вы можете использовать маскировку целых слов, заменив приведенный выше data_collator на соответствующий. В этом случае вам нужно будет удалить первую строку здесь:

In [ ]:
downsampled_dataset = downsampled_dataset.remove_columns(["word_ids"])
eval_dataset = downsampled_dataset["test"].map(
    insert_random_mask,
    batched=True,
    remove_columns=downsampled_dataset["test"].column_names,
)
eval_dataset = eval_dataset.rename_columns(
    {
        "masked_input_ids": "input_ids",
        "masked_attention_mask": "attention_mask",
        "masked_labels": "labels",
    }
)

Затем мы можем настроить загрузчики данных как обычно, но для тестового набора мы будем использовать default_data_collator из  Transformers:

In [ ]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

batch_size = 64
train_dataloader = DataLoader(
    downsampled_dataset["train"],
    shuffle=True,
    batch_size=batch_size,
    collate_fn=data_collator,
)
eval_dataloader = DataLoader(
    eval_dataset, batch_size=batch_size, collate_fn=default_data_collator
)

Далее мы выполняем стандартные действия с помощью Accelerate. В первую очередь необходимо загрузить новую версию предварительно подготовленной модели:

In [ ]:
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

Затем нам нужно указать оптимизатор; мы будем использовать стандартный AdamW:

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

С помощью этих объектов мы теперь можем подготовить все для обучения с помощью объекта Accelerator:

In [ ]:
from accelerate import Accelerator

accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

Теперь, когда наша модель, оптимизатор и загрузчики данных настроены, мы можем указать планировщик скорости обучения следующим образом:


In [ ]:
from transformers import get_scheduler

num_train_epochs = 3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

Локальная директория для сохранения моделей (необязательно, можно сразу в HuggingFace сохранять)

In [ ]:
output_dir = "/content/drive/MyDrive/NLP_models/distilbert-base-uncased-finetuned-imdb-accelerate"

Полный цикл обучения и оценки модели с помощью метрики perplexity:

In [ ]:
from tqdm.auto import tqdm
import torch
import math

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    # Training
    model.train()
    for batch in train_dataloader:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    losses = []
    for step, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            outputs = model(**batch)

        loss = outputs.loss
        losses.append(accelerator.gather(loss.repeat(batch_size)))

    losses = torch.cat(losses)
    losses = losses[: len(eval_dataset)]
    try:
        perplexity = math.exp(torch.mean(losses))
    except OverflowError:
        perplexity = float("inf")

    print(f">>> Epoch {epoch}: Perplexity: {perplexity}")

    # Save
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
    if accelerator.is_main_process:
        tokenizer.save_pretrained(output_dir)

Замените TestingStudent123 на свой ник в HuggingFace. Мы создаём репозиторий для нашей обученной модели, чтобы иметь возможность использовать её в дальнейшем.

In [ ]:
unwrapped_model.push_to_hub("TestingStudent123/distilbert-base-uncased-finetuned-imdb-accelerate",
            commit_message=f"Training finished {epoch}", blocking=False
        )

### Использование обученной модели

Вы можете взаимодействовать с вашей моделью либо с помощью ее виджета на HuggingFace, либо локально с помощью pipeline из Transformers. Давайте воспользуемся последним для загрузки нашей модели с помощью pipeline:
"fill-mask" - задача, которую решаем

In [ ]:
from transformers import pipeline

mask_filler = pipeline(
    "fill-mask", model="distilbert-base-uncased-finetuned-imdb-accelerate"
)

Теперь мы можем скормить pipeline наш примерный текст “This is a great [MASK]” и посмотреть, каковы 5 лучших прогнозов:

In [ ]:
preds = mask_filler(text)

for pred in preds:
    print(f">>> {pred['sequence']}")

Отлично — наша модель явно адаптировала свои весовые коэффициенты для прогнозирования слов, которые сильнее ассоциируются с фильмами!